# Week 8 Demo — Offline AI Log Analysis Pipeline (End-to-End)

**Deliverable:** document Section 4, Week 8 — *"End-to-end pipeline integration + full evaluation on test set — Deliverable: Evaluation report + demo notebook"*.

This notebook demonstrates the **complete offline pipeline** on one sample log, wiring together every frozen component built in Weeks 1–7 without modifying any of them:

| Step | Component | Built in |
|---|---|---|
| 1. Load retriever | `src/rag/rag_retriever.py` + `data/faiss.index` | Week 7 |
| 2. Load fine-tuned model | `src/training/evaluate.py` loader + Week 5 LoRA adapter | Weeks 4–6 |
| 3. Preprocess a raw log | `src/preprocessing/preprocessor.py` | Week 2 |
| 4. Retrieve top-3 incidents | FAISS cosine search over 1,550 train incidents | Week 7 |
| 5. Inject context | `inject_context()` context-injection pipeline | Week 7 |
| 6. Generate response | greedy decoding, same settings as all evaluations | Weeks 4–6 |
| 7. Parse output | `src/training/output_parser.py` | Week 4 |
| 8. Display final analysis | structured incident analysis | — |

This is exactly the Section 5.6 pipeline order: `normalized = preprocessor.process(logs)` → `context = retriever.retrieve(normalized, top_k)` → `model.generate(normalized, context=...)` → `parse_model_output(result)`.

**Runs 100% offline** — local model weights, local embedder, local FAISS index; no network access is attempted.

> Run from the repository root kernel (or let the setup cell below fix `sys.path`). Requires the offline ML venv (torch, transformers, peft, faiss, sentence-transformers) and the Week 5 adapter + Week 7 index on disk. A CUDA GPU is strongly recommended (CPU generation on the 3B model takes ~30–60 s).

## Step 0 — Offline setup

Offline environment variables are set **before** any Hugging Face import (document Section 2: zero internet dependencies at runtime), and the repository root is put on `sys.path` so `src.*` imports work from `notebooks/`.

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

import sys
import json
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():          # kernel started in notebooks/
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

DEVICE = "cuda"   # switch to "cpu" if no GPU (slow: ~30-60 s/generation)

print(f"Repository root : {REPO_ROOT}")
print(f"HF_HUB_OFFLINE  : {os.environ['HF_HUB_OFFLINE']}")
print(f"Device          : {DEVICE}")

Repository root : C:\Users\Manmay\Downloads\log-analysis-slm
HF_HUB_OFFLINE  : 1
Device          : cuda


## Step 1 — Load the offline RAG retriever (Week 7)

Loads the persisted FAISS index (`data/faiss.index`, 1,550 vectors — **train split only**, so nothing the model is tested on was ever indexed) and its incident-text sidecar, plus the local `all-MiniLM-L6-v2` sentence embedder from `models/sentence-transformers/` (document Section 5.5).

In [2]:
from src.rag.rag_retriever import (
    DEFAULT_INDEX_FILE,
    DEFAULT_SIDECAR_FILE,
    OfflineRAGRetriever,
    inject_context,
)

retriever = OfflineRAGRetriever()          # local embedder, offline
retriever.load(DEFAULT_INDEX_FILE, DEFAULT_SIDECAR_FILE)

print(f"FAISS index    : {DEFAULT_INDEX_FILE.name} "
      f"({retriever.index.ntotal} vectors, dim {retriever.index.d})")
print(f"Incident store : {len(retriever.incidents)} "
      "(text, metadata) records — train.jsonl only")

c:\Users\Manmay\Downloads\log-analysis-slm\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2441.00it/s]


FAISS index    : faiss.index (1550 vectors, dim 384)
Incident store : 1550 (text, metadata) records — train.jsonl only


## Step 2 — Load the fine-tuned model (Weeks 4–6)

Reuses the **frozen** loader from `src/training/evaluate.py` (imported, not modified): Qwen2.5-3B-Instruct from local weights, with the Week 5 QLoRA adapter attached via PEFT and merged for inference — identical to how the Week 6 evaluation loaded it.

In [3]:
from src.training.evaluate import (
    DEFAULT_ADAPTER_DIR,
    DEFAULT_MODEL_DIR,
    GENERATION_KWARGS,
    build_prompt,
    load_model_and_tokenizer,
)

model, tokenizer = load_model_and_tokenizer(
    DEFAULT_MODEL_DIR, DEVICE, adapter_dir=DEFAULT_ADAPTER_DIR
)
print(f"Model loaded on {DEVICE} with adapter "
      f"{DEFAULT_ADAPTER_DIR.name} (merged for inference)")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:08<00:00, 48.35it/s]


Loading LoRA adapter from C:\Users\Manmay\Downloads\log-analysis-slm\models\checkpoints\final-adapter (offline)...


W0719 23:18:47.683000 21692 venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Model loaded on cuda with adapter final-adapter (merged for inference)


## Step 3 — Preprocess one sample log (Week 2)

A raw multi-line log excerpt (as it would arrive from a log file) is normalized with the Week 2 preprocessor: variable values — IPs, timestamps, ports, file paths, memory addresses — are replaced with placeholders, and lines are grouped into a log window.

In [4]:
from src.preprocessing.preprocessor import extract_windows

RAW_LOG = """\
2026-07-18 09:14:02 kernel: EXT4-fs error (device sda1): ext4_find_entry:1455: inode #131077: comm smartd: reading directory lblock 0
2026-07-18 09:14:02 smartd[1024]: Device: /dev/sda [SAT], 8 Currently unreadable (pending) sectors
2026-07-18 09:14:03 smartd[1024]: Device: /dev/sda [SAT], SMART Prefailure Attribute: 5 Reallocated_Sector_Ct changed from 100 to 97
2026-07-18 09:14:05 kernel: sd 0:0:0:0: [sda] tag#29 FAILED Result: hostbyte=DID_OK driverbyte=DRIVER_SENSE
2026-07-18 09:14:05 kernel: sd 0:0:0:0: [sda] tag#29 Sense Key : Medium Error [current]
2026-07-18 09:14:05 kernel: blk_update_request: critical medium error, dev sda, sector 52428800
"""

raw_lines = [l for l in RAW_LOG.splitlines() if l.strip()]
windows = extract_windows(raw_lines)      # normalize + window (Week 2)
normalized_log = "\n".join(windows[0])

print(f"{len(raw_lines)} raw lines -> {len(windows)} normalized window(s)\n")
print("--- Normalized log window (model input) ---")
print(normalized_log)

6 raw lines -> 1 normalized window(s)

--- Normalized log window (model input) ---
<TIMESTAMP> kernel: EXT4-fs error (device sda1): ext4_find_entry:<PORT>: inode #131077: comm smartd: reading directory lblock 0
<TIMESTAMP> smartd[1024]: Device: /dev/sda [SAT], 8 Currently unreadable (pending) sectors
<TIMESTAMP> smartd[1024]: Device: /dev/sda [SAT], SMART Prefailure Attribute: 5 Reallocated_Sector_Ct changed from 100 to 97
<TIMESTAMP> kernel: sd 0:0:0:0: [sda] tag#29 FAILED Result: hostbyte=DID_OK driverbyte=DRIVER_SENSE
<TIMESTAMP> kernel: sd 0:0:0:0: [sda] tag#29 Sense Key : Medium Error [current]
<TIMESTAMP> kernel: blk_update_request: critical medium error, dev sda, sector 52428800


## Step 4 — Retrieve the top-3 similar past incidents (Week 7)

The normalized window is embedded with the local sentence transformer and searched against the FAISS index (cosine similarity). This is the Section 5.6 call: `context = retriever.retrieve(normalized, top_k=3)`.

In [5]:
retrieved = retriever.retrieve_with_metadata(normalized_log, top_k=3)

for n, (text, meta) in enumerate(retrieved, 1):
    print(f"{n}. [{meta.get('severity')} — {meta.get('incident_type')}] "
          f"(train #{meta.get('train_index')})")
    print("   " + " ".join(text.split())[:160] + "...\n")

1. [HIGH — Disk I/O Failure] (train #1306)
   <TIMESTAMP> ERROR [kernel] I/O error, dev sda, sector 48291712 <TIMESTAMP> ERROR [kernel] Buffer I/O error on device sda, logical block 6036464 <TIMESTAMP> WARN...

2. [HIGH — Disk I/O Failure] (train #207)
   <TIMESTAMP> ERROR [kernel] I/O error, dev nvme0n1, sector 90118244 <TIMESTAMP> ERROR [kernel] Buffer I/O error on device nvme0n1, logical block 11264780 <TIMEST...

3. [MEDIUM — Supercomputer Node Error] (train #754)
   - 1131567043 2005.11.09 tbird-admin1 Nov 9 12:10:43 local@tbird-admin1 scsi0 : LSI Logic MegaRAID driver - 1131567043 2005.11.09 tbird-admin1 Nov 9 12:10:43 loc...



## Step 5 — Inject retrieved context (Week 7)

The Week 7 context-injection pipeline prepends the retrieved incidents to the log window. The result feeds the existing prompt builder **unchanged** — this augmented input is the only difference versus a plain fine-tuned inference.

In [6]:
augmented_input = inject_context(normalized_log, retrieved)
print(augmented_input)

SIMILAR PAST INCIDENTS (retrieved offline via FAISS):
1. [HIGH — Disk I/O Failure] <TIMESTAMP> ERROR [kernel] I/O error, dev sda, sector 48291712 <TIMESTAMP> ERROR [kernel] Buffer I/O error on device sda, logical block 6036464 <TIMESTAMP> WARN [smartd] Device sda: 137 reallocated sectors (was 24)
2. [HIGH — Disk I/O Failure] <TIMESTAMP> ERROR [kernel] I/O error, dev nvme0n1, sector 90118244 <TIMESTAMP> ERROR [kernel] Buffer I/O error on device nvme0n1, logical block 11264780 <TIMESTAMP> WARN [smartd] Device nvme0n1: 52 reallocated sectors (was 3)
3. [MEDIUM — Supercomputer Node Error] - 1131567043 2005.11.09 tbird-admin1 Nov 9 12:10:43 local@tbird-admin1 scsi0 : LSI Logic MegaRAID driver - 1131567043 2005.11.09 tbird-admin1 Nov 9 12:10:43 local@tbird-admin1 scsi[0]: scanning scsi channel 0 [Phy 0] for non-raid devices - 1131567043 2005.11.09 tbird-admin1 Nov 9 12:10:43 local@tbird-admin1 scsi[0]: scanning scsi channel 1 [virtual] for logical drives - 1131567043 2005.11.09 tbird

CURREN

## Step 6 — Generate the analysis (Weeks 4–6 harness)

The prompt is built with the same chat template and system instruction used across all evaluations (taken from the dataset's `instruction` field), and generated with the same **greedy, deterministic** settings (`GENERATION_KWARGS`) as the Week 4/6/8 evaluations.

In [ ]:
import time
import torch

# Same system instruction the model was trained/evaluated with.
with open(REPO_ROOT / "data" / "dataset" / "test.jsonl",
          encoding="utf-8") as fh:
    instruction = json.loads(fh.readline())["instruction"]

prompt = build_prompt(tokenizer, instruction, augmented_input)
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

gen_kwargs = {k: v for k, v in GENERATION_KWARGS.items()
              if v is not None}                      # greedy decoding

start = time.perf_counter()
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        pad_token_id=tokenizer.eos_token_id,
        **gen_kwargs,
    )
latency = time.perf_counter() - start

generated = output_ids[0][inputs["input_ids"].shape[1]:]
raw_response = tokenizer.decode(generated, skip_special_tokens=True)

print(f"Generated in {latency:.1f}s (greedy, offline)\n")
print("--- Raw model output ---")
print(raw_response)

## Step 7 — Parse the structured output (Week 4)

The **frozen** output parser extracts the structured fields (severity, incident type, root cause, summary, recommended actions) — the same parser used to score every evaluation.

In [ ]:
from src.training.output_parser import parse_model_output

analysis = parse_model_output(raw_response)

print(f"parse_errors: {analysis['parse_errors'] or 'none'}")
for key in ("severity", "incident_type", "root_cause", "summary"):
    print(f"{key:15s}: {analysis[key]}")

## Step 8 — Final analysis

The complete, structured incident analysis — produced end-to-end on this machine with zero network access.

In [ ]:
sep = "=" * 62
print(sep)
print("OFFLINE LOG ANALYSIS — FINAL RESULT")
print(sep)
print(f"Severity        : {analysis['severity']}")
print(f"Incident type   : {analysis['incident_type']}")
print(f"Root cause      : {analysis['root_cause']}")
print(f"Summary         : {analysis['summary']}")
print("Recommended actions:")
for i, action in enumerate(analysis["recommended_actions"] or [], 1):
    print(f"  {i}. {action}")
print(sep)
print("Similar past incidents used as context:")
for n, (_, meta) in enumerate(retrieved, 1):
    print(f"  {n}. {meta.get('severity')} — {meta.get('incident_type')} "
          f"(train #{meta.get('train_index')})")
print(sep)
print(f"Generation latency: {latency:.1f}s | "
      f"Parse errors: {len(analysis['parse_errors'])} | "
      "Network access: none")

---

## What this demonstrated

- **Every component ran from local files**: model weights (`models/qwen25-3b`), LoRA adapter (`models/checkpoints/final-adapter`), embedder (`models/sentence-transformers/all-MiniLM-L6-v2`), FAISS index (`data/faiss.index`). `HF_HUB_OFFLINE=1` / `TRANSFORMERS_OFFLINE=1` were set before any import — no network access was possible.
- **No Week 1–7 code was modified** — the notebook only *imports* the frozen preprocessor, evaluation harness, parser, and retriever.
- The same pipeline, run over the full 198-example test set, produces the **D6 evaluation report**: `python -m src.rag.evaluate_rag --device cuda` → `reports/Week8_Evaluation_Report.md` (baseline vs fine-tuned vs RAG comparison, confusion matrices, ROUGE scores).
- Week 9 wraps these exact calls in the FastAPI `/analyze` endpoint (document Section 5.6).